# DINO-to-image decoder (RAE) interpolation -- qualitative claim only, no number

Backs a single qualitative sentence in `boundaries.tex` ("DINO-to-image decoder"); no macro, no table row.

**Needs:** DTD, a DINO-to-image (RAE) decoder checkpoint.

**Maintained runnable path:** none needed -- no quantitative claim to reproduce.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = os.environ.get('DATA_ROOT', "/content/drive/MyDrive/DATA_ROOT")  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
dtd_root = os.path.join(PROJECT_DIR, "dtd/images")
OUT = os.path.join(PROJECT_DIR, "featup_interp"); os.makedirs(OUT, exist_ok=True)

!pip install -q git+https://github.com/mhamilton723/FeatUp pot scikit-learn
# If FeatUp complains about kornia/torch, that's expected on fresh Colab; usually resolves itself.

In [ ]:
import torch, numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

upsampler = torch.hub.load("mhamilton723/FeatUp", "dinov2", use_norm=True).to(device).eval()

from PIL import Image
import torchvision.transforms as T
_norm = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

def load_img(path, size=224):
    # FeatUp DINOv2 wants multiples of 14; 224 -> dense 224x224 field is fine
    im = Image.open(path).convert("RGB").resize((size,size), Image.BICUBIC)
    x = T.ToTensor()(im).unsqueeze(0).to(device)
    return x

def first_img(cat):
    fs = sorted(f for f in os.listdir(os.path.join(dtd_root,cat))
                if f.lower().endswith((".jpg",".jpeg",".png")))
    return os.path.join(dtd_root, cat, fs[0])

@torch.no_grad()
def featup_dense(img01):
    x = _norm(img01)
    hr = upsampler(x)          # [1, 384, H, W] dense
    lr = upsampler.model(x)    # [1, 384, 16, 16] coarse (the raw DINO grid)
    return hr, lr

hr, lr = featup_dense(load_img(first_img("dotted")))
print("dense:", tuple(hr.shape), "| coarse:", tuple(lr.shape))   # expect [1,384,224,224], [1,384,16,16]

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def feat_pca_rgb(field):  # [1,384,H,W] -> [H,W,3] for viewing
    C,H,W = field.shape[1:]
    f = field[0].reshape(C,-1).T.cpu().numpy()
    rgb = PCA(3).fit_transform(f)
    rgb = (rgb - rgb.min(0)) / (np.ptp(rgb, axis=0) + 1e-9)
    return rgb.reshape(H,W,3)

fig,ax=plt.subplots(1,4,figsize=(13,3.3))
for i,cat in enumerate(["dotted","banded","honeycombed","chequered"]):
    hr,_ = featup_dense(load_img(first_img(cat)))
    ax[i].imshow(feat_pca_rgb(hr)); ax[i].set_title(cat); ax[i].axis("off")
plt.show()
# GATE: if these show structured, texture-aligned features (dots/stripes visible in PCA-RGB) -> FeatUp works on DTD.

In [ ]:
import ot as pot
from scipy.optimize import linear_sum_assignment

ETAS = np.linspace(0,1,11)
N_SUB = 4096   # subsample for OT, exactly like your CNT pixel experiment

def field_to_points(field):
    C = field.shape[1]
    return field[0].reshape(C,-1).T            # [H*W, 384]

def path_curve_linear(PA, PB):
    """normalized feature-distance from A along linear path (per-point corresponding)."""
    dAB = (PA-PB).pow(2).mean().sqrt()+1e-9
    return np.array([((PA - ((1-e)*PA+e*PB)).pow(2).mean().sqrt()/dAB).item() for e in ETAS])

def ot_targets(PA, PB, reg=0.05):
    """Hungarian on a 4096 subsample, NN-extend to all points (your CNT recipe)."""
    n = PA.shape[0]
    ia = torch.randperm(n)[:N_SUB]; ib = torch.randperm(n)[:N_SUB]
    a_, b_ = PA[ia], PB[ib]
    C = torch.cdist(a_,b_).pow(2).cpu().numpy()
    row,col = linear_sum_assignment(C)
    tgt_sub = b_[torch.as_tensor(col, device=PA.device)]   # [N_SUB,384]
    # NN-extend: each full point -> target of nearest subsampled source
    d = torch.cdist(PA, a_)               # [n, N_SUB]
    nn = d.argmin(1)
    return tgt_sub[nn]                     # [n,384]

def path_curve_ot(PA, PB):
    tgt = ot_targets(PA, PB)
    dAB = (PA-PB).pow(2).mean().sqrt()+1e-9
    return np.array([((PA - ((1-e)*PA+e*tgt)).pow(2).mean().sqrt()/dAB).item() for e in ETAS])

def run_pair(ca, cb):
    hrA,_ = featup_dense(load_img(first_img(ca)))
    hrB,_ = featup_dense(load_img(first_img(cb)))
    PA, PB = field_to_points(hrA), field_to_points(hrB)
    cl, co = path_curve_linear(PA,PB), path_curve_ot(PA,PB)
    return cl[5], co[5]   # midpoints; want closeness to 0.5

print(run_pair("banded","dotted"))

In [ ]:
from scipy.stats import wilcoxon
import random; random.seed(0)
cats = sorted(c for c in os.listdir(dtd_root) if os.path.isdir(os.path.join(dtd_root,c)))
pairs=[]
while len(pairs)<40:
    a,b=random.choice(cats),random.choice(cats)
    if a!=b: pairs.append((a,b))

L,O=[],[]
for ca,cb in pairs:
    try:
        ml,mo = run_pair_gram(ca,cb); L.append(abs(ml-0.5)); O.append(abs(mo-0.5))
    except Exception as e: print(f"skip {ca}/{cb}: {e}")
L,O=np.array(L),np.array(O)
print(f"pairs {len(L)} | |mid-0.5| LIN {L.mean():.3f} OT {O.mean():.3f} | OT wins {(O<L).sum()}/{len(L)}")
print(f"Wilcoxon p={wilcoxon(L,O,alternative='greater').pvalue:.2e}")

In [ ]:
def gram(P):  # P: [n, 384] point set -> 384x384 Gram (normalized)
    Pc = P - P.mean(0, keepdim=True)
    return (Pc.T @ Pc) / P.shape[0]

def path_curve_gram(PA, PB, targets=None):
    """Distance in GRAM space (nonlinear in features) — not tautological."""
    GA, GB = gram(PA), gram(PB)
    dAB = (GA-GB).pow(2).mean().sqrt()+1e-9
    out=[]
    for e in ETAS:
        mid = (1-e)*PA + e*(PB if targets is None else targets)
        out.append(((GA-gram(mid)).pow(2).mean().sqrt()/dAB).item())
    return np.array(out)

def run_pair_gram(ca, cb):
    hrA,_=featup_dense(load_img(first_img(ca))); hrB,_=featup_dense(load_img(first_img(cb)))
    PA,PB=field_to_points(hrA),field_to_points(hrB)
    cl=path_curve_gram(PA,PB)
    co=path_curve_gram(PA,PB,targets=ot_targets(PA,PB))
    return cl[5],co[5]

In [ ]:
# In the WORKING RAE notebook. Does the encoder accept larger images -> more tokens?
for size in [224, 448, 896]:
    try:
        img = load_img(first_img("dotted"), size=size)
        z = rae_encode(img)
        print(f"input {size}: latent {tuple(z.shape)}")
        rec = rae_decode(z)
        print(f"   recon {tuple(rec.shape)}, range [{rec.min():.2f},{rec.max():.2f}]")
    except Exception as e:
        print(f"input {size}: FAILED — {type(e).__name__}: {str(e)[:120]}")

In [ ]:
import os
PROJECT_DIR = os.environ.get('DATA_ROOT', "/content/drive/MyDrive/DATA_ROOT")  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
dtd_root = os.path.join(PROJECT_DIR, "dtd/images")
RAE_OUT = os.path.join(PROJECT_DIR, "rae_interp"); os.makedirs(RAE_OUT, exist_ok=True)

# Clone + install RAE
%cd /content
!rm -rf RAE
!git clone -q https://github.com/bytetriper/RAE.git
%cd /content/RAE
!pip install -q omegaconf einops timm scipy scikit-learn pot
!pip install -q -r requirements.txt
# Pin the two deps that RAE needs and FeatUp broke:
!pip install -q "numpy>=1.26,<2.0" "huggingface_hub>=0.34.0,<1.0"

# Download just the DINOv2-B decoder + its normalization stats
!hf download nyu-visionx/RAE-collections --local-dir /content/RAE/models \
  --include "decoders/dinov2/wReg_base/ViTXL_n08/*" "stats/dinov2/wReg_base/imagenet1k/*"

print("\n>>> NOW: Runtime > Restart session. Then run the cells below (do NOT re-run this cell).")

In [ ]:
import os, sys, torch, numpy as np
PROJECT_DIR = os.environ.get('DATA_ROOT', "/content/drive/MyDrive/DATA_ROOT")  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
dtd_root = os.path.join(PROJECT_DIR, "dtd/images")
RAE_OUT = os.path.join(PROJECT_DIR, "rae_interp")

%cd /content/RAE
sys.path.insert(0, "/content/RAE")
sys.path.insert(0, "/content/RAE/src")

from utils.model_utils import instantiate_from_config
from utils.train_utils import parse_configs
from stage1 import RAE

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

CONFIG = "/content/RAE/configs/stage1/pretrained/DINOv2-B.yaml"
rae_config, *_ = parse_configs(CONFIG)
assert rae_config is not None, "no stage_1 section parsed"
rae = instantiate_from_config(rae_config).to(device).eval()
print("RAE ready")

In [ ]:
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

def load_img(path, size=256):
    im = Image.open(path).convert("RGB").resize((size, size), Image.BICUBIC)
    return transforms.ToTensor()(im).unsqueeze(0).to(device)   # [1,3,size,size] in [0,1]

def first_img(cat):
    fs = sorted(f for f in os.listdir(os.path.join(dtd_root,cat))
                if f.lower().endswith((".jpg",".jpeg",".png")))
    return os.path.join(dtd_root, cat, fs[0])

@torch.no_grad()
def rae_encode(img01): return rae.encode(img01)

@torch.no_grad()
def rae_decode(z):     return rae.decode(z).clamp(0,1)

def to_np(img01): return img01[0].detach().cpu().permute(1,2,0).numpy()

def show_row(imgs, titles, fname=None):
    n=len(imgs); plt.figure(figsize=(3.2*n,3.2))
    for i,(im,t) in enumerate(zip(imgs,titles)):
        plt.subplot(1,n,i+1); plt.imshow(np.clip(to_np(im),0,1)); plt.title(t,fontsize=9); plt.axis("off")
    if fname: plt.savefig(os.path.join(RAE_OUT,fname),dpi=130,bbox_inches="tight")
    plt.show()

In [ ]:
for size in [224, 448, 896]:
    try:
        img = load_img(first_img("dotted"), size=size)
        z = rae_encode(img)
        print(f"input {size}: latent {tuple(z.shape)}")
        rec = rae_decode(z)
        print(f"   recon {tuple(rec.shape)}, range [{rec.min():.2f},{rec.max():.2f}]")
    except Exception as e:
        print(f"input {size}: FAILED — {type(e).__name__}: {str(e)[:140]}")

In [ ]:
# In the FeatUp notebook. Tiny per-image feature->RGB MLP, overfit to A, then decode midpoints.
import torch, torch.nn as nn, numpy as np

def fit_tiny_decoder(field, img01, steps=300):
    C,H,W = field.shape[1:]
    X = field[0].reshape(C,-1).T.detach()                      # frozen feature inputs
    Y = torch.nn.functional.interpolate(img01,(H,W),mode="bilinear")[0].reshape(3,-1).T.detach()
    dec = nn.Sequential(nn.Linear(C,256), nn.GELU(),
                        nn.Linear(256,256), nn.GELU(),
                        nn.Linear(256,3)).to(field.device)
    opt = torch.optim.Adam(dec.parameters(), lr=1e-3)
    with torch.enable_grad():
        for _ in range(steps):
            opt.zero_grad()
            loss = ((dec(X) - Y)**2).mean()                    # grad flows through dec's params only
            loss.backward()
            opt.step()
    return dec, (H,W)

# Fit on A, decode A, linear-mid, OT-mid
ca,cb = "banded","dotted"
A,B = load_img(first_img(ca)), load_img(first_img(cb))
fa,_ = featup_dense(A); fb,_ = featup_dense(B)
dec,HW = fit_tiny_decoder(fa, A)
PA,PB = field_to_points(fa), field_to_points(fb)
lin_mid = (0.5*PA+0.5*PB).T.reshape(1,384,*HW)
ot_tgt  = ot_targets(PA,PB)
ot_mid  = (0.5*PA+0.5*ot_tgt).T.reshape(1,384,*HW)

import matplotlib.pyplot as plt
imgs = [A, decode_field(dec,fa,HW), decode_field(dec,lin_mid,HW), decode_field(dec,ot_mid,HW)]
tt   = ["A real","A decoded","LINEAR mid","OT mid"]
plt.figure(figsize=(13,3.3))
for i,(im,t) in enumerate(zip(imgs,tt)):
    plt.subplot(1,4,i+1); plt.imshow(np.clip(im[0].detach().cpu().permute(1,2,0).numpy(),0,1)); plt.title(t); plt.axis("off")
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = os.environ.get('DATA_ROOT', "/content/drive/MyDrive/DATA_ROOT")  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
dtd_root = os.path.join(PROJECT_DIR, "dtd/images")
OUT = os.path.join(PROJECT_DIR, "featup_interp"); os.makedirs(OUT, exist_ok=True)

!pip install -q git+https://github.com/mhamilton723/FeatUp pot scikit-learn
print("If pip changed numpy/torch, Runtime > Restart session, then run the cells below.")

In [ ]:
import torch, numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

upsampler = torch.hub.load("mhamilton723/FeatUp", "dinov2", use_norm=True).to(device).eval()

from PIL import Image
import torchvision.transforms as T
from scipy.optimize import linear_sum_assignment
_norm = T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

def load_img(path, size=224):
    im = Image.open(path).convert("RGB").resize((size,size), Image.BICUBIC)
    return T.ToTensor()(im).unsqueeze(0).to(device)

def first_img(cat):
    fs = sorted(f for f in os.listdir(os.path.join(dtd_root,cat))
                if f.lower().endswith((".jpg",".jpeg",".png")))
    return os.path.join(dtd_root, cat, fs[0])

@torch.no_grad()
def featup_dense(img01):
    x = _norm(img01)
    return upsampler(x), upsampler.model(x)     # dense [1,384,H,W], coarse [1,384,16,16]

def field_to_points(field):
    C = field.shape[1]
    return field[0].reshape(C,-1).T              # [HW, 384]

N_SUB = 4096
def ot_targets(PA, PB):
    n = PA.shape[0]
    ia = torch.randperm(n)[:N_SUB]; ib = torch.randperm(n)[:N_SUB]
    a_, b_ = PA[ia], PB[ib]
    C = torch.cdist(a_,b_).pow(2).cpu().numpy()
    row,col = linear_sum_assignment(C)
    tgt_sub = b_[torch.as_tensor(col, device=PA.device)]
    nn = torch.cdist(PA, a_).argmin(1)
    return tgt_sub[nn]

_hr,_ = featup_dense(load_img(first_img("dotted")))
print("dense field:", tuple(_hr.shape))          # confirm [1,384,224,224]

In [ ]:
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
# ViT-B is the smallest SAM; plenty for feature extraction, ~375MB
!wget -q -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -P /content
import os
print("ckpt:", os.path.exists("/content/sam_vit_b_01ec64.pth"))

In [ ]:
import torch, numpy as np
from segment_anything import sam_model_registry
device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry["vit_b"](checkpoint="/content/sam_vit_b_01ec64.pth").to(device).eval()
sam_encoder = sam.image_encoder
for p in sam_encoder.parameters(): p.requires_grad_(False)

# SAM expects 1024x1024, normalized with its own pixel mean/std
SAM_MEAN = torch.tensor([123.675,116.28,103.53], device=device).view(1,3,1,1)
SAM_STD  = torch.tensor([58.395,57.12,57.375], device=device).view(1,3,1,1)

from PIL import Image
import torchvision.transforms as T

def load_img_sam(path, size=1024):
    im = Image.open(path).convert("RGB").resize((size,size), Image.BICUBIC)
    x = T.ToTensor()(im).unsqueeze(0).to(device) * 255.0     # SAM wants [0,255]
    return (x - SAM_MEAN) / SAM_STD

@torch.no_grad()
def sam_dense(path):
    x = load_img_sam(path)
    feat = sam_encoder(x)          # [1, 256, 64, 64]
    return feat

f = sam_dense(first_img("dotted"))
print("SAM feature grid:", tuple(f.shape))   # expect [1, 256, 64, 64] -> 4096 tokens, 256-dim

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def feat_pca_rgb(field):
    C,H,W = field.shape[1:]
    f = field[0].reshape(C,-1).T.cpu().numpy()
    rgb = PCA(3).fit_transform(f)
    rgb = (rgb - rgb.min(0)) / (np.ptp(rgb, axis=0)+1e-9)
    return rgb.reshape(H,W,3)

fig,ax=plt.subplots(1,4,figsize=(13,3.3))
for i,cat in enumerate(["dotted","banded","honeycombed","chequered"]):
    ax[i].imshow(feat_pca_rgb(sam_dense(first_img(cat)))); ax[i].set_title(f"SAM: {cat}"); ax[i].axis("off")
plt.show()
# GATE: visible texture structure (dots/stripes/grid) -> SAM features carry texture info, OT test is meaningful.

In [ ]:
import ot as pot
from scipy.optimize import linear_sum_assignment
from scipy.stats import wilcoxon
import random; random.seed(0)

ETAS = np.linspace(0,1,11)
N_SUB = 4096   # SAM gives exactly 4096 tokens at 64x64, so this is effectively full-OT

def field_to_points(field):
    C = field.shape[1]; return field[0].reshape(C,-1).T

def ot_targets(PA, PB):
    n = PA.shape[0]
    ia = torch.randperm(n)[:N_SUB]; ib = torch.randperm(n)[:N_SUB]
    a_,b_ = PA[ia], PB[ib]
    C = torch.cdist(a_,b_).pow(2).cpu().numpy()
    row,col = linear_sum_assignment(C)
    tgt_sub = b_[torch.as_tensor(col, device=PA.device)]
    nn = torch.cdist(PA, a_).argmin(1)
    return tgt_sub[nn]

def gram(P):
    Pc = P - P.mean(0, keepdim=True)
    return (Pc.T @ Pc) / P.shape[0]

def path_curve_gram(PA, PB, targets=None):
    GA,GB = gram(PA), gram(PB)
    dAB = (GA-GB).pow(2).mean().sqrt()+1e-9
    out=[]
    for e in ETAS:
        mid = (1-e)*PA + e*(PB if targets is None else targets)
        out.append(((GA-gram(mid)).pow(2).mean().sqrt()/dAB).item())
    return np.array(out)

def run_pair_gram(ca, cb):
    PA = field_to_points(sam_dense(first_img(ca)))
    PB = field_to_points(sam_dense(first_img(cb)))
    cl = path_curve_gram(PA,PB)
    co = path_curve_gram(PA,PB,targets=ot_targets(PA,PB))
    return cl[5], co[5]

cats = sorted(c for c in os.listdir(dtd_root) if os.path.isdir(os.path.join(dtd_root,c)))
pairs=[]
while len(pairs)<40:
    a,b=random.choice(cats),random.choice(cats)
    if a!=b: pairs.append((a,b))

L,O=[],[]
for ca,cb in pairs:
    try:
        ml,mo = run_pair_gram(ca,cb); L.append(abs(ml-0.5)); O.append(abs(mo-0.5))
    except Exception as e: print(f"skip {ca}/{cb}: {e}")
L,O=np.array(L),np.array(O)
print(f"SAM | pairs {len(L)} | |mid-0.5| LIN {L.mean():.3f} OT {O.mean():.3f} | OT wins {(O<L).sum()}/{len(L)}")
print(f"Wilcoxon p={wilcoxon(L,O,alternative='greater').pvalue:.2e}")

In [ ]:
import torch.nn as nn, matplotlib.pyplot as plt

def fit_tiny_decoder_sam(field, img_path, steps=300):
    C,H,W = field.shape[1:]
    X = field[0].reshape(C,-1).T.detach()
    img = T.ToTensor()(Image.open(img_path).convert("RGB").resize((H,W),Image.BICUBIC)).unsqueeze(0).to(device)
    Y = img[0].reshape(3,-1).T.detach()
    dec = nn.Sequential(nn.Linear(C,256), nn.GELU(), nn.Linear(256,256), nn.GELU(), nn.Linear(256,3)).to(device)
    opt = torch.optim.Adam(dec.parameters(), lr=1e-3)
    with torch.enable_grad():
        for _ in range(steps):
            opt.zero_grad(); loss=((dec(X)-Y)**2).mean(); loss.backward(); opt.step()
    return dec, (H,W)

def decode_field(dec, field, HW):
    C=field.shape[1]; return dec(field[0].reshape(C,-1).T).clamp(0,1).T.reshape(1,3,*HW)

def probe_sam(ca, cb):
    fa, fb = sam_dense(first_img(ca)), sam_dense(first_img(cb))
    dec,HW = fit_tiny_decoder_sam(fa, first_img(ca))
    PA,PB = field_to_points(fa), field_to_points(fb)
    lin = (0.5*PA+0.5*PB).T.reshape(1,256,*HW)
    ot_ = (0.5*PA+0.5*ot_targets(PA,PB)).T.reshape(1,256,*HW)

    A_disp = T.ToTensor()(Image.open(first_img(ca)).convert("RGB").resize(HW)).unsqueeze(0).to(device)
    imgs = [A_disp, decode_field(dec,fa,HW), decode_field(dec,lin,HW), decode_field(dec,ot_,HW)]
    tt   = ["A real","A decoded","LINEAR mid","OT mid"]
    plt.figure(figsize=(13,3.3))
    for i,(im,t) in enumerate(zip(imgs,tt)):
        plt.subplot(1,4,i+1); plt.imshow(np.clip(im[0].detach().cpu().permute(1,2,0).numpy(),0,1)); plt.title(f"{ca}/{cb}: {t}"); plt.axis("off")
    plt.show()

for ca,cb in [("banded","dotted"),("honeycombed","marbled"),("chequered","striped")]:
    probe_sam(ca,cb)

for ca,cb in [("banded","dotted"),("honeycombed","marbled"),("chequered","striped")]:
    probe_sam(ca,cb)

In [ ]:
import torch.nn as nn, matplotlib.pyplot as plt, numpy as np, random

# ---- decoded-image texture-plausibility score (independent of the OT metric) ----
import torchvision.models as tvm
_vgg = tvm.vgg16(weights=tvm.VGG16_Weights.DEFAULT).features[:16].to(device).eval()
for p in _vgg.parameters(): p.requires_grad_(False)
_vmean=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1)
_vstd =torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)

@torch.no_grad()
def vgg_gram(img01):
    x=(img01-_vmean)/_vstd
    f=_vgg(torch.nn.functional.interpolate(x,224,mode="bilinear"))
    f=f.flatten(2); return (f@f.transpose(1,2))/f.shape[-1]

# Build a small bank of REAL decoded textures to measure "distance to nearest real"
def build_real_bank(cats, per=1, HW=(64,64)):
    bank=[]
    for c in cats:
        p=first_img(c)
        img=T.ToTensor()(Image.open(p).convert("RGB").resize(HW)).unsqueeze(0).to(device)
        bank.append(vgg_gram(img))
    return bank

def dist_to_nearest_real(img01, bank):
    g=vgg_gram(img01)
    return min(((g-gb).pow(2).mean().item()) for gb in bank)

# ---- per-pair probe that returns the plausibility scores ----
def probe_sam_score(ca, cb, bank, show=False):
    fa, fb = sam_dense(first_img(ca)), sam_dense(first_img(cb))
    dec,HW = fit_tiny_decoder_sam(fa, first_img(ca))
    PA,PB = field_to_points(fa), field_to_points(fb)
    lin = (0.5*PA+0.5*PB).T.reshape(1,256,*HW)
    ot_ = (0.5*PA+0.5*ot_targets(PA,PB)).T.reshape(1,256,*HW)
    lin_img, ot_img = decode_field(dec,lin,HW), decode_field(dec,ot_,HW)
    d_lin = dist_to_nearest_real(lin_img, bank)
    d_ot  = dist_to_nearest_real(ot_img, bank)
    if show:
        A_disp=T.ToTensor()(Image.open(first_img(ca)).convert("RGB").resize(HW)).unsqueeze(0).to(device)
        for im,t in zip([A_disp,decode_field(dec,fa,HW),lin_img,ot_img],
                        ["A","A dec",f"LIN d={d_lin:.2f}",f"OT d={d_ot:.2f}"]):
            pass
        plt.figure(figsize=(13,3.2))
        for i,(im,t) in enumerate(zip([A_disp,decode_field(dec,fa,HW),lin_img,ot_img],
                                      ["A real","A decoded",f"LIN (d={d_lin:.2f})",f"OT (d={d_ot:.2f})"])):
            plt.subplot(1,4,i+1); plt.imshow(np.clip(im[0].detach().cpu().permute(1,2,0).numpy(),0,1)); plt.title(f"{ca}/{cb}: {t}",fontsize=8); plt.axis("off")
        plt.show()
    return d_lin, d_ot

# ---- sweep: 12 pairs, mixed; lower distance-to-real = more on-manifold = better ----
random.seed(1)
cats = sorted(c for c in os.listdir(dtd_root) if os.path.isdir(os.path.join(dtd_root,c)))
bank = build_real_bank(cats)                          # one real per category as the reference manifold
pairs=[]
while len(pairs)<12:
    a,b=random.choice(cats),random.choice(cats)
    if a!=b and (a,b) not in pairs: pairs.append((a,b))

DL,DO=[],[]
for k,(ca,cb) in enumerate(pairs):
    try:
        dl,do = probe_sam_score(ca,cb,bank, show=(k<6))   # show first 6 strips, score all 12
        DL.append(dl); DO.append(do)
        print(f"{ca:12s}/{cb:12s}  LIN {dl:.2f}  OT {do:.2f}  {'OT better' if do<dl else 'lin better'}")
    except Exception as e:
        print(f"skip {ca}/{cb}: {e}")

from scipy.stats import wilcoxon
DL,DO=np.array(DL),np.array(DO)
print(f"\n=== decoded texture-plausibility (dist to nearest real, LOWER=on-manifold) ===")
print(f"pairs {len(DL)} | LIN {DL.mean():.2f}  OT {DO.mean():.2f} | OT more on-manifold {(DO<DL).sum()}/{len(DL)}")
try: print(f"Wilcoxon (LIN more off-manifold than OT): p={wilcoxon(DL,DO,alternative='greater').pvalue:.2e}")
except Exception as e: print("wilcoxon:",e)

In [ ]:
random.seed(7)
pairs=[]
while len(pairs)<40:
    a,b=random.choice(cats),random.choice(cats)
    if a!=b and (a,b) not in pairs: pairs.append((a,b))

DL,DO,GAP,SEP=[],[],[],[]
for ca,cb in pairs:
    try:
        fa,fb = sam_dense(first_img(ca)), sam_dense(first_img(cb))
        PA,PB = field_to_points(fa), field_to_points(fb)
        sep = (PA.mean(0)-PB.mean(0)).pow(2).sum().sqrt().item()   # A<->B structural distance
        dl,do = probe_sam_score(ca,cb,bank, show=False)
        DL.append(dl); DO.append(do); GAP.append(dl-do); SEP.append(sep)
    except Exception as e: print(f"skip {ca}/{cb}: {e}")

DL,DO,GAP,SEP=map(np.array,(DL,DO,GAP,SEP))
from scipy.stats import wilcoxon, spearmanr
print(f"pairs {len(DL)} | LIN {DL.mean():.2f} OT {DO.mean():.2f} | OT on-manifold {(DO<DL).sum()}/{len(DL)}")
print(f"median: LIN {np.median(DL):.2f} OT {np.median(DO):.2f}  (robust to outliers)")
print(f"Wilcoxon p={wilcoxon(DL,DO,alternative='greater').pvalue:.2e}")
rho,pr = spearmanr(SEP, GAP)
print(f"OT-advantage vs A-B structural distance: Spearman rho={rho:.2f} p={pr:.2e}  (positive => OT helps more when textures differ more)")